<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Regresión Logística — La balanza de probabilidades y el umbral de decisión
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/01_Regresion_Logistica_Simple_y_Multiple_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Aprenderás a usar la **Regresión Logística** — el clasificador más fundamental de ML — sin fórmulas intimidantes:

1. Cómo entrenar el modelo con Scikit-Learn.
2. La diferencia entre `predict_proba()` (probabilidad) y `predict()` (etiqueta).
3. Cómo ajustar el **umbral de decisión** (no siempre el 50% es el mejor).
4. Cómo interpretar los coeficientes del modelo.

---
## 1. La balanza de decisiones ⚖️

Piensa en la Regresión Logística como una **balanza** que pesa cada factor de riesgo:

```
            ¿Tiene cardiopatía?
                   ⚖️
   FACTORES DE RIESGO    |    FACTORES PROTECTORES
   ✚ Edad alta           |    ✔ Colesterol bajo
   ✚ Presión alta        |    ✔ Frecuencia cardíaca alta
   ✚ Dolor de pecho      |    ✔ Sin angina
```

El modelo asigna un **"peso" (coeficiente)** a cada factor. Si la suma de los pesos positivos supera a los negativos, el paciente es clasificado como en riesgo.

- Un coeficiente **positivo** → ese factor **aumenta** la probabilidad de la clase 1.
- Un coeficiente **negativo** → ese factor **disminuye** la probabilidad de la clase 1.

In [ ]:
import os, urllib.parse, urllib.request, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def load_dataset(filename, module_name="08 - Classification"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    return target_path

df = pd.read_csv(load_dataset('heart_disease.csv'))
target_col = df.columns[-1]
print(f"Dataset: {df.shape[0]} pacientes · {df.shape[1]-1} variables · objetivo: '{target_col}'")
print(df[target_col].value_counts().rename({0:'Sin cardiopatía (0)', 1:'Con cardiopatía (1)'}))

In [ ]:
# Preparar datos
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Escalar (importante para regresión logística)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenar el modelo
modelo = LogisticRegression(random_state=42, max_iter=1000)
modelo.fit(X_train_sc, y_train)

print(f"✅ Modelo entrenado con {len(X_train)} pacientes")
print(f"   Exactitud en datos de prueba: {accuracy_score(y_test, modelo.predict(X_test_sc)):.1%}")

---
## 2. `predict_proba` vs `predict` — Probabilidad vs Etiqueta 🎲

In [ ]:
# predict_proba da la PROBABILIDAD (un número entre 0 y 1)
probs = modelo.predict_proba(X_test_sc)

# predict da la ETIQUETA (0 o 1, según umbral 0.5 por defecto)
etiquetas = modelo.predict(X_test_sc)

# Mostramos los primeros 10 pacientes del conjunto de prueba
comparacion = pd.DataFrame({
    'P(sin cardiopatía)': probs[:, 0].round(3),
    'P(con cardiopatía)': probs[:, 1].round(3),
    'Predicción': etiquetas,
    'Real': y_test.values
}).head(10)

comparacion['¿Correcto?'] = (comparacion['Predicción'] == comparacion['Real']).map({True:'✅', False:'❌'})
print("🔍 Primeros 10 pacientes — probabilidades vs etiquetas:")
display(comparacion)

print("\n💡 La columna 'P(con cardiopatía)' es el porcentaje de riesgo que calcula el modelo.")
print("   Si ese porcentaje > 50% → el modelo predice cardiopatía.")

---
## 3. Los coeficientes — ¿Qué factores pesan más? ⚖️

In [ ]:
coefs = pd.Series(modelo.coef_[0], index=X.columns).sort_values()

plt.figure(figsize=(10, 5))
colores = ['#ef4444' if c > 0 else '#10b981' for c in coefs]
bars = plt.barh(coefs.index, coefs.values, color=colores, alpha=0.85, edgecolor='white')
plt.axvline(0, color='black', linewidth=1.5)
plt.title('Coeficientes de la Regresión Logística\n(rojo = aumenta riesgo, verde = reduce riesgo)',
          fontweight='bold', fontsize=12)
plt.xlabel('Peso del coeficiente (escala estandarizada)')
plt.tight_layout()
plt.show()

print("\n💡 Los factores con barras rojas hacia la derecha AUMENTAN el riesgo de cardiopatía.")
print("   Los factores con barras verdes hacia la izquierda REDUCEN el riesgo.")
print(f"\n   Mayor factor de riesgo:    '{coefs.idxmax()}' ({coefs.max():.3f})")
print(f"   Mayor factor protector:   '{coefs.idxmin()}' ({coefs.min():.3f})")

---
## 4. Ajustar el umbral — No siempre 50% es lo mejor 🎛️

El umbral del 50% significa: "Si el modelo cree que hay más de 50% de probabilidad → clasifica como positivo".

Pero en medicina, a veces preferimos ser más conservadores:
- **Umbral bajo (ej. 30%)** → detectamos más casos positivos, pero también más falsas alarmas.
- **Umbral alto (ej. 70%)** → solo reportamos casos cuando estamos muy seguros, pero nos perdemos algunos reales.

> 💡 **Analogía:** La prueba de embarazo. Si el umbral es muy bajo (muy sensible), da positivo con trazas mínimas (pocas falsas negativas pero más falsas positivas). Si el umbral es alto, solo da positivo con niveles altos (más seguro pero puede perderse casos tempranos).

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

probs_positivas = probs[:, 1]
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Umbral':>8} | {'Exactitud':>10} | {'Recall':>8} | {'Precisión':>10} | {'F1':>6}")
print("-" * 55)
for u in umbrales:
    pred_u = (probs_positivas >= u).astype(int)
    acc = accuracy_score(y_test, pred_u)
    rec = recall_score(y_test, pred_u, zero_division=0)
    pre = precision_score(y_test, pred_u, zero_division=0)
    f1  = f1_score(y_test, pred_u, zero_division=0)
    marca = " ← estándar" if u == 0.5 else ""
    print(f"   {u:.1f}   |   {acc:.3f}     |  {rec:.3f}  |   {pre:.3f}    | {f1:.3f}{marca}")

print("\n💡 En medicina, preferiríamos un recall alto (detectar todos los enfermos)")
print("   aunque eso signifique algunas falsas alarmas. → Umbral más bajo.")

---
## 5. Resumen 🎓

- ✅ La **Regresión Logística** predice probabilidades entre 0% y 100% usando la función Sigmoide.
- ✅ `predict_proba()` → número (probabilidad); `predict()` → etiqueta (0 o 1).
- ✅ Los **coeficientes** indican qué variables aumentan o reducen el riesgo.
- ✅ El **umbral** (por defecto 50%) se puede ajustar según el costo de los errores.

> 🚀 **Siguiente paso:** Ve al cuaderno `02_Clasificacion_Multiclase_y_Fronteras_Decision_Dummies.ipynb`.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>